**Upload the merged dataset**

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving master_raw_Notebook1.csv to master_raw_Notebook1.csv


**Import Pandas and read the dataset**

In [ ]:
import pandas as pd

master = pd.read_csv("master_raw_Notebook1.csv")

print(master.shape)

(36191, 3)


**Check the Python version**

In [ ]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


**Install RDKit**

In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 52.6 MB/s eta 0:00:00


**Verify the RDKit installation**

In [ ]:
from rdkit import Chem
print(Chem.rdBase.rdkitVersion)

2026.03.3


**Load the dataset again**

In [ ]:
import pandas as pd

master = pd.read_csv("master_raw_Notebook1.csv")
master.head()

,SMILES,Label,Source
0,O=Cc1ccco1,0,OASIS
1,O=Cc1ccco1,1,OASIS
2,O=Cc1ccco1,0,OASIS
3,O=Cc1ccco1,0,OASIS
4,O=Cc1ccco1,0,OASIS


**Import RDKit standardization modules**

In [ ]:
from rdkit import Chem
from rdkit.Chem import MolStandardize

**Define the SMILES standardization function**

In [ ]:
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # Cleanup
        mol = rdMolStandardize.Cleanup(mol)

        # Largest fragment
        chooser = rdMolStandardize.LargestFragmentChooser()
        mol = chooser.choose(mol)

        # Uncharge
        uncharger = rdMolStandardize.Uncharger()
        mol = uncharger.uncharge(mol)

        # Canonical SMILES
        return Chem.MolToSmiles(mol, canonical=True)

    except:
        return None

**Standardize all SMILES**

In [ ]:
master["Standard_SMILES"] = master["SMILES"].apply(standardize_smiles)

Streaming output truncated to the last 5000 lines.
[13:08:25] Running MetalDisconnector
[13:08:25] Initializing Normalizer
[13:08:25] Running Normalizer
[13:08:25] Running LargestFragmentChooser
[13:08:25] Running Uncharger
[13:08:25] Initializing MetalDisconnector
[13:08:25] Running MetalDisconnector
[13:08:25] Initializing Normalizer
[13:08:25] Running Normalizer
[13:08:25] Running LargestFragmentChooser
[13:08:25] Running Uncharger
[13:08:25] Initializing MetalDisconnector
[13:08:25] Running MetalDisconnector
[13:08:25] Initializing Normalizer
[13:08:25] Running Normalizer
[13:08:25] Running LargestFragmentChooser
[13:08:25] Running Uncharger
[13:08:25] Initializing MetalDisconnector
[13:08:25] Running MetalDisconnector
[13:08:25] Initializing Normalizer
[13:08:25] Running Normalizer
[13:08:25] Running LargestFragmentChooser
[13:08:25] Running Uncharger
[13:08:25] Initializing MetalDisconnector
[13:08:25] Running MetalDisconnector
[13:08:25] Initializing Normalizer
[13:08:25] Runnin

**Remove invalid standardized molecules**

In [ ]:
master = master.dropna(subset=["Standard_SMILES"]).reset_index(drop=True)

**Check the standardized dataset**


In [ ]:
print(master.shape)
master.head()

(36190, 4)


,SMILES,Label,Source,Standard_SMILES
0,O=Cc1ccco1,0,OASIS,O=Cc1ccco1
1,O=Cc1ccco1,1,OASIS,O=Cc1ccco1
2,O=Cc1ccco1,0,OASIS,O=Cc1ccco1
3,O=Cc1ccco1,0,OASIS,O=Cc1ccco1
4,O=Cc1ccco1,0,OASIS,O=Cc1ccco1


**Count duplicate standardized molecules**

In [ ]:
master["Standard_SMILES"].duplicated().sum()

np.int64(26726)

**Count unique standardized molecules**

In [ ]:
master["Standard_SMILES"].nunique()

9464

**Detect conflicting labels**

In [ ]:
conflicts = (
    master.groupby("Standard_SMILES")["Label"]
    .nunique()
    .reset_index()
)

conflicts = conflicts[conflicts["Label"] > 1]

print("Conflicting molecules:", len(conflicts))

Conflicting molecules: 842


**Remove conflicting molecules**

In [ ]:
conflict_smiles = conflicts["Standard_SMILES"]

master_clean = master[
    ~master["Standard_SMILES"].isin(conflict_smiles)
]

print(master_clean.shape)

(29438, 4)


**Remove duplicate molecules**

In [ ]:
master_clean = master_clean.drop_duplicates(
    subset="Standard_SMILES",
    keep="first"
)

print(master_clean.shape)

(8622, 4)


**Verify the cleaned dataset**

In [ ]:
print("Unique molecules:", master_clean["Standard_SMILES"].nunique())
print("Duplicates:", master_clean["Standard_SMILES"].duplicated().sum())

Unique molecules: 8622
Duplicates: 0


**Save the cleaned dataset**

In [ ]:
master_clean.to_csv("master_clean_standardized.csv", index=False)

In [ ]:
from google.colab import files
files.download("master_clean_standardized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>